In [1]:
# Install dependencies
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Create an API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"


In [10]:
# helper functions
from anthropic.types import Message


def add_user_message(messages, message):  
    user_message = {
      "role": "user", 
      "content": message.content if isinstance(message, Message) else message
      }
    messages.append(user_message)


def add_assistant_message(messages, message):  
    assistant_message = {
      "role": "assistant", 
      "content": message.content if isinstance(message, Message) else message
      }
    messages.append(assistant_message)




def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):

  params = {
    "model": model,
    "max_tokens": 1000,
    "messages": messages,
    "temperature": temperature,
    "stop_sequences": stop_sequences,
    "tools": tools
  }
  
  if tools:
    params["tools"] = tools
  
  if system:
    params["system"] = system
  if stop_sequences:
    params["stop_sequences"] = stop_sequences


  message = client.messages.create( **params )


  return message


  def text_from_message(message):
    return "\n".join(
      [block.text for block in message.content if block.type == "text"]
    )

In [11]:
from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [12]:
from anthropic.types import ToolParam
# Tools and Schemas

from datetime import datetime, timedelta

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam( {
  
  "name": "get_current_datetime",
  "description": "Returns the current local date and time as a formatted string. Use this tool whenever you need to know the present date or time, such as for timestamping, scheduling, calculating durations relative to now, or answering questions about the current day, date, or time. The output is formatted using Python strftime-style directives.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime format string that controls how the current datetime is rendered. Uses standard strftime directives, e.g. '%Y-%m-%d %H:%M:%S' produces '2025-06-19 14:30:00', '%Y-%m-%d' produces just the date, and '%H:%M' produces just the hour and minute. If omitted, defaults to '%Y-%m-%d %H:%M:%S'. Must be a non-empty string; passing an empty string will cause an error."
      }
    },
    "required": []
  }
})


def add_duration_to_datetime(
  datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
  ):
  date = datetime.strptime(datetime_str, input_format)


In [13]:
import json


def run_tool(tool_name, tool_input):
  if tool_name == "get_current_datetime":
    return get_current_datetime(**tool_input)
  elif tool_name == "add_duration_to_datetime":
    return add_duration_to_datetime(**tool_input)
  elif tool_name == "set_reminder":
    return set_reminder(**tool_input)
  elif tool_name == "batch_tool":
    return batch_tool(**tool_input)
  else:
    raise ValueError(f"Tool {tool_name} not found")



def run_tools(message):
  tool_request = [
    block for block in message.content if block.type == "tool_use"
  ]

  tool_result_blocks = []

  for tool_request in tool_requests:
    
      try:
        tool_output = run_tool(tool_request.name, tool_request.input)
        tool_result_block = {
          "type": "tool_result",
          "tool_use_id": tool_request.id,
          "content": json.dumps(tool_output),
          "is_error": False,
        }
      except Exception as e:
        tool_result_block = {
          "type": "tool_result",
          "tool_use_id": tool_request.id,
          "content": f"Error: {e}",
          "is_error": True,
        }
      
      tool_result_blocks.append(tool_result_block)
  
  return tool_result_blocks
      

In [14]:
def run_conversation(messages):
  while True:
    response = chat(messages, tools=[get_current_datetime_schema,
    add_duration_to_datetime_schema,
    set_reminder_schema,
    batch_tool_schema 
    ])

    add_assistant_message(messages, response)
    print(text_from_message(response))
    
    if response.stop_reason != "tool_use":
      break
    
    tool_results = run_tools(response)
    add_user_message(messages, tool_results)
  
  return messages


In [15]:
messages = []
add_user_message(
  messages, 
  "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
)
run_conversation(messages)


NameError: name 'text_from_message' is not defined

In [16]:
# Cell 0: client
from dotenv import load_dotenv
from anthropic import Anthropic
load_dotenv()
client = Anthropic()
model = "claude-haiku-4-5"

# Cell 1: helpers
from anthropic.types import Message

def add_user_message(messages, message):
    messages.append({"role": "user", "content": message.content if isinstance(message, Message) else message})

def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content if isinstance(message, Message) else message})

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {"model": model, "max_tokens": 1000, "messages": messages,
              "temperature": temperature, "stop_sequences": stop_sequences}
    if tools: params["tools"] = tools
    if system: params["system"] = system
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

# Cell 2: tools + schemas
from datetime import datetime, timedelta

def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds": new_date = date + timedelta(seconds=duration)
    elif unit == "minutes": new_date = date + timedelta(minutes=duration)
    elif unit == "hours": new_date = date + timedelta(hours=duration)
    elif unit == "days": new_date = date + timedelta(days=duration)
    elif unit == "weeks": new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12; year -= 1
        day = min(date.day, [31, 29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                             31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years": new_date = date.replace(year=date.year + duration)
    else: raise ValueError(f"Unsupported time unit: {unit}")
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")

def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format.",
    "input_schema": {"type": "object", "properties": {
        "datetime_str": {"type": "string", "description": "The input datetime string."},
        "duration": {"type": "number", "description": "Amount of time to add."},
        "unit": {"type": "string", "description": "seconds, minutes, hours, days, weeks, months, or years."},
        "input_format": {"type": "string", "description": "strptime format, default %Y-%m-%d."}},
        "required": ["datetime_str"]}}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that notifies the user at the specified time with the provided content.",
    "input_schema": {"type": "object", "properties": {
        "content": {"type": "string", "description": "The reminder message text."},
        "timestamp": {"type": "string", "description": "ISO 8601 timestamp YYYY-MM-DDTHH:MM:SS."}},
        "required": ["content", "timestamp"]}}

# Cell 3: get_current_datetime
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format string.",
    "input_schema": {"type": "object", "properties": {
        "date_format": {"type": "string", "description": "strftime format. Default %Y-%m-%d %H:%M:%S.",
                        "default": "%Y-%m-%d %H:%M:%S"}}, "required": []}})

# Cell 4: run_tool / run_tools (CORRECTED version)
import json

def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime": return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime": return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder": return set_reminder(**tool_input)

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []
    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_blocks.append({"type": "tool_result", "tool_use_id": tool_request.id,
                                       "content": json.dumps(tool_output), "is_error": False})
        except Exception as e:
            tool_result_blocks.append({"type": "tool_result", "tool_use_id": tool_request.id,
                                       "content": f"Error: {e}", "is_error": True})
    return tool_result_blocks

# Cell 5: run_conversation
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema,
                                         add_duration_to_datetime_schema, set_reminder_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        if response.stop_reason != "tool_use":
            break
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    return messages

print("setup cells 0-5 loaded — run_tools refreshed with corrected (tool_requests) version ✓")

setup cells 0-5 loaded — run_tools refreshed with corrected (tool_requests) version ✓


In [18]:
messages = []
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.",
)
run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. Let me first calculate when 177 days after January 1st, 2050 is.
Now I'll set a reminder for your doctor's appointment on June 27, 2050:
----
Setting the following reminder for 2050-06-27T12:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. The reminder is now scheduled and you'll be notified at that time.


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. Let me first calculate when 177 days after January 1st, 2050 is.", type='text'),
   ToolUseBlock(id='toolu_01TjQgC5mZqR9oUgaimtSz9X', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01TjQgC5mZqR9oUgaimtSz9X',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050:", type='text'),
   ToolUseBlock(id='toolu_01SUWZfQUARzFcDzz2YD44iA', caller=DirectCaller(type='direct'), input={'content': "Doctor's ap